In [0]:
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
from src.instacart.serving import validate_recommendation_export

# --------------------------------------------------
# 1. Start Spark and Databricks utilities
# --------------------------------------------------

spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)


# --------------------------------------------------
# 2. Databricks source table
# --------------------------------------------------

source_table = (
    "workspace.ml_data."
    "next_basket_serving_recommendations"
)


# --------------------------------------------------
# 3. Neon PostgreSQL connection
# --------------------------------------------------

neon_host = (
    "ep-summer-sea-ag9wnzfy-pooler."
    "c-2.eu-central-1.aws.neon.tech"
)

neon_port = "5432"
neon_database = "neondb"
neon_user = "neondb_owner"

neon_table = "next_basket_recommendations"


# --------------------------------------------------
# 4. Get password from Databricks Secrets
# --------------------------------------------------

neon_password = dbutils.secrets.get(
    scope="neon",
    key="password"
)


# --------------------------------------------------
# 5. Load final recommendations from Databricks
# --------------------------------------------------

recommendations_df = spark.table(
    source_table
)

recommendation_count = recommendations_df.count()

print(
    "Recommendations to export:",
    recommendation_count
)

validate_recommendation_export(
    recommendations_df.columns,
    recommendation_count,
)

print("Pre-export validation passed.")
# --------------------------------------------------
# 6. Export recommendations to Neon PostgreSQL
# --------------------------------------------------

(
    recommendations_df
    .write
    .format("postgresql")
    .option("host", neon_host)
    .option("port", neon_port)
    .option("database", neon_database)
    .option("dbtable", neon_table)
    .option("user", neon_user)
    .option("password", neon_password)
    .option("batchsize", "5000")
    .option("numPartitions", "2")
    .mode("overwrite")
    .save()
)


# --------------------------------------------------
# 7. Confirmation
# --------------------------------------------------

print(
    "Export completed:",
    neon_table
)

print(
    "Rows exported:",
    recommendation_count
)

Recommendations to export: 131800
Export completed: next_basket_recommendations
Rows exported: 131800
